# Exercise 4

## Question 1
Create tables to keep track of trading positions

**Hint**

After each trade, we want to keep at least the following information of each traded instrument:
1. the date and time when the trade takes place
2. the name of the instrument traded
3. the quantity of the instrument that we possess after the trade
4. The amount of cash available to us for trading the instrument.

In [ ]:
import numpy as np
from scipy.stats import norm


class GBM:
    def __init__(self):
        self.mu = np.nan
        self.sigma = np.nan
        self.rng = np.random.default_rng()

    def calibrate(self, trajectory, Dt):
        increments = np.diff(np.log(trajectory))
        moments = [0, 0]
        n_iter = 10
        for iter in range(n_iter):
            X = self.rng.choice(increments, size=len(increments) // 2)
            moments[0] += np.mean(X) / n_iter
            moments[1] += np.mean(X**2) / n_iter
        std = np.sqrt(moments[1] - moments[0] ** 2)
        self.sigma = std / np.sqrt(Dt)
        self.mu = moments[0] / Dt + self.sigma**2 / 2

    def forecast(self, S0, t, confidence):
        predicted = S0 * np.exp(self.mu * t)
        mu = (self.mu - self.sigma**2 / 2) * t
        sigma = self.sigma * np.sqrt(t)
        log_return_low, log_return_high = norm.ppf(
            [(1 - confidence) / 2, (1 + confidence) / 2], loc=mu, scale=sigma
        )
        price_low = S0 * np.exp(log_return_low)
        price_high = S0 * np.exp(log_return_high)
        return {
            "confidence": confidence,
            "expected": predicted,
            "interval": [price_low, price_high],
        }

In [ ]:
import sqlite3
import csv
from contextlib import closing

# Load SP500.csv into the database
with closing(sqlite3.connect("./../SP500.db")) as load_conn:
    lcs = load_conn.cursor()
    lcs.execute("""
    create table if not exists prices (
        theday text primary key,
        price real
    );
    """)
    with closing(open("./../SP500.csv")) as datafile:
        reader = csv.DictReader(datafile, fieldnames=["date", "price"], delimiter="\t")
        for row in reader:
            lcs.execute(
                "insert or replace into prices values (?, ?)",
                (row["date"], float(row["price"])),
            )
    load_conn.commit()
print(f"Prices loaded into DB")

In [ ]:
import sqlite3
from contextlib import closing

# Keep connection open for the whole session
conn = sqlite3.connect("./../SP500.db")
cs = conn.cursor()

cs.execute("""
create table if not exists positions (
    time_of_trade text,
    instrument text,
    quantity real,
    cash real,
    primary key (time_of_trade, instrument)
);
""")
conn.commit()
print("Database ready")

In [ ]:
# Seed the positions table with starting capital before any trades
cs.execute("delete from positions;")
cs.execute(
    "insert or replace into positions values ('2020-01-01', 'SP500', 0, 100000.0);"
)
conn.commit()
print("Positions table seeded: 0 shares, $100,000 cash")

## Question 2
Choose a lookback time of 120 days and perform backtesting on the prices from 2021-05-02 to 2021-05-31.

**hint**

1. On each day from 2021-05-02 to 2021-05-31, take the latest 120 days' prices before this date and calibrate a GBM model.
2. Use the GBM model to forecast the price in 10 days:
   $$S_{t + n\Delta t} = S_t \exp \left[
   \left(
   \mu - \frac{\sigma^2}{2}
   \right)n \Delta t
   + W_{t + n \Delta t} - W_t
   \right]$$
The price forecast is
$$
\mathbb E S_{t + n\Delta t} = S_t \exp \left(
   \mu n \Delta t
\right)
$$
In this case, $n=10, \Delta t = 1/250$. Use your code from question 3 of exercise 3 to estimate the confidence interval of the forecast.

3. The relative return of buying the instrument on day 0 and selling it on day 10 is approximately normally distributed with mean $(\mu - \frac{\sigma^2}{2}) n \Delta t$ and standard deviation $\sigma \sqrt{n \Delta t}$. Calculate the 95% expected shortfall $ES_\alpha$ in 10 days:
$$ES_\alpha = -m + s \frac{\phi(\Phi^{-1}(\alpha))}{1 - \alpha}$$
where $\alpha = 0.95$ and
$$m = \left(\mu - \frac{\sigma^2}{2} \right) n \Delta t$$
$$s = \sigma \sqrt{n \Delta t}$$
$\phi$ and $\Phi^{-1}$ are the density and the quantile functions of the standard normal distribution, respectively. You can calculate them in Python using scipy.stats.norm.pdf and scipy.stats.norm.ppf, respectively.

In [ ]:
from scipy.stats import norm

mu = 0.0511
sigma = 0.1440
alpha = 0.95
n = 10
Dt = 1 / 250
m = (mu - sigma**2 / 2) * n * Dt
s = sigma * np.sqrt(n * Dt)

alpha = 0.95
ES = -m + s * norm.pdf(norm.ppf(alpha)) / (1 - alpha)
print(f"The 95% expected shortfall is {ES:.4f}")

4. Devise your buy/sell signal based on the forecasted price and its confidence interval. For example, if your forecast suggests that you can be 80% sure that the price in 10 days will be higher than the current price, buy the instrument.

In [ ]:
def position_size(which_day, forecast):
    cs.execute(f"""
    select quantity, cash from positions
    where instrument = 'SP500'
    and time_of_trade < '{which_day}'
    order by time_of_trade desc
    limit 1;
    """)
    qty, cash = cs.fetchall()[0]
    cs.execute(f"""
    select price from prices
    where theday <= '{which_day}'
    order by theday desc
    limit 1;
    """)
    price = cs.fetchall()[0][0]
    capital = cash + qty * price
    if price < forecast["interval"][0]:
        return round(capital / price)
    elif price > forecast["interval"][1]:
        return -round(capital / price)
    else:
        return 0

5. Decide your risk appetite in terms of the *Expected Shortfall*. For example, if you decide that your risk appetite is 95% expected shortfall being 5%, then the ratio of capital that you should use for the trade is $\frac{0.05}{ES_{0.95}}$. If this figure is larger than 1, you may consider using leverage.

In [ ]:
def position_size(which_day, forecast, ES):
    cs.execute(f"""
    select quantity, cash from positions
    where instrument = 'SP500'
    and time_of_trade < '{which_day}'
    order by time_of_trade desc
    limit 1;
    """)
    qty, cash = cs.fetchall()[0]
    cs.execute(f"""
    select price from prices
    where theday <= '{which_day}'
    order by theday desc
    limit 1;
    """)
    price = cs.fetchall()[0][0]
    capital = cash + qty * price
    exposure = capital * 0.05 / ES
    if price < forecast["interval"][0]:
        return round(exposure / price)
    elif price > forecast["interval"][1]:
        return -round(exposure / price)
    else:
        return 0

6. Follow your buy/sell signals through the said time period. Store in the database the number of the instruments that you hold and the amount of your cash reserve after each trade.

In [ ]:
def analyse(which_day):
    cs.execute(f"""
    select price from prices where theday <= '{which_day}'
    order by theday desc limit 120;
    """)
    P = np.flipud(np.asarray(cs.fetchall())).flatten()
    model = GBM()
    Dt = 1.0 / 252
    model.calibrate(P, Dt)
    confidence = 0.1
    n = 10
    T = n * Dt
    forecast = model.forecast(P[-1], T, confidence)

    m = (model.mu - model.sigma**2 / 2) * n * Dt
    s = model.sigma * np.sqrt(n * Dt)

    alpha = 0.95
    ES = -m + s * norm.pdf(norm.ppf(alpha)) / (1 - alpha)
    return position_size(which_day, forecast, ES)


def main(begin_on):
    cs.execute(f"select theday from prices where theday >= '{begin_on}';")
    days = [d[0] for d in cs.fetchall()]
    asset = {"old": np.nan, "new": np.nan}
    cash = {"old": np.nan, "new": np.nan}
    cs.execute("delete from positions where time_of_trade > '2020-01-01';")

    print(
        f"\n{'Date':<12} {'Action':>6} {'Price':>8} {'Qty':>6} {'Cash':>12} {'Capital':>12}"
    )
    print("-" * 60)

    for d in days:
        asset["new"] = analyse(d)
        cs.execute(f"""
        select quantity, cash from positions
        where time_of_trade < '{d}'
        order by time_of_trade desc
        limit 1;
        """)
        asset["old"], cash["old"] = cs.fetchall()[0]
        cs.execute(f"""
        select price from prices
        where theday <= '{d}'
        order by theday desc
        limit 1;
        """)
        latest = cs.fetchall()[0][0]
        trade_size = round(asset["new"]) - round(asset["old"])
        if trade_size != 0:
            cash["new"] = cash["old"] - trade_size * latest
            cs.execute(f"""
            insert into positions values
            ('{d}', 'SP500', {round(asset['new'])}, {cash['new']});
            """)
            action = "BUY" if trade_size > 0 else "SELL"
            capital = cash["new"] + round(asset["new"]) * latest
        else:
            cash["new"] = cash["old"]
            action = "HOLD"
            capital = cash["old"] + round(asset["old"]) * latest

        print(
            f"{d:<12} {action:>6} {latest:>8.2f} {round(asset['new']):>6} {cash['new']:>12.2f} {capital:>12.2f}"
        )
        conn.commit()

In [ ]:
# Cell: run the backtest
main("2021-05-02")

In [ ]:
# Summary: show all recorded positions
print("\n--- All trades recorded in DB ---")
cs.execute("""
    select time_of_trade, instrument, quantity, cash 
    from positions 
    order by time_of_trade;
""")
rows = cs.fetchall()
print(f"\n{'Date':<12} {'Instrument':<12} {'Quantity':>10} {'Cash':>12}")
print("-" * 50)
for row in rows:
    print(f"{row[0]:<12} {row[1]:<12} {row[2]:>10.0f} {row[3]:>12.2f}")

# Final P&L
cs.execute("""
    select quantity, cash from positions 
    order by time_of_trade desc limit 1;
""")
final_qty, final_cash = cs.fetchone()
cs.execute("""
    select price from prices 
    order by theday desc limit 1;
""")
final_price = cs.fetchone()[0]
final_capital = final_cash + final_qty * final_price

print(f"\n--- Final Summary ---")
print(f"Final quantity held : {final_qty:.0f} shares")
print(f"Final cash          : ${final_cash:.2f}")
print(f"Final price         : ${final_price:.2f}")
print(f"Final capital       : ${final_capital:.2f}")
print(f"Starting capital    : $100,000.00")
print(
    f"P&L                 : ${final_capital - 100000:.2f} ({(final_capital/100000 - 1)*100:.2f}%)"
)

In [ ]:
# Cell: close the connection cleanly when done
conn.close()
print("Done")